In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path
import sys

project_dir = Path("/content/drive/MyDrive/Colab Notebooks/nlp_translation")

if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

from src.data import load_prepared, make_loaders

config, vocab_fr, vocab_en, prepared = load_prepared(project_dir)

# Réduire la mémoire utilisée
config["batch_size"] = 8

loaders = make_loaders(prepared, config)

train_loader = loaders["train"]
validation_loader = loaders["validation"]
test_loader = loaders["test"]

print("Entraînement :", len(prepared["train"]))
print("Validation :", len(prepared["validation"]))
print("Test :", len(prepared["test"]))
print("Batch size :", config["batch_size"])

Entraînement : 30000
Validation : 2000
Test : 2000
Batch size : 8


In [3]:
print("Dossier du projet :", project_dir)
print("Phrases d'entraînement :", len(prepared["train"]))
print("Phrases de validation :", len(prepared["validation"]))
print("Phrases de test :", len(prepared["test"]))
print("Vocabulaire français :", len(vocab_fr))
print("Vocabulaire anglais :", len(vocab_en))
print("Batch size :", config["batch_size"])

batch = next(iter(train_loader))

print("\nDimensions d'un batch :")
for name in ["src", "src_lengths", "decoder_input", "target"]:
    print(name, ":", tuple(batch[name].shape))

Dossier du projet : /content/drive/MyDrive/Colab Notebooks/nlp_translation
Phrases d'entraînement : 30000
Phrases de validation : 2000
Phrases de test : 2000
Vocabulaire français : 10000
Vocabulaire anglais : 10000
Batch size : 8

Dimensions d'un batch :
src : (8, 39)
src_lengths : (8,)
decoder_input : (8, 58)
target : (8, 58)


# **1. Imports et choix du GPU**

In [4]:
import random
import torch
import torch.nn as nn

from torch.nn.utils.rnn import pack_padded_sequence

PAD_ID = config["special_tokens"]["<PAD>"]

random.seed(config["seed"])
torch.manual_seed(config["seed"])

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Appareil utilisé :", device)

Appareil utilisé : cuda


# **2. Construire l’encodeur, le décodeur et le modèle complet**

In [5]:
from torch.nn.utils.rnn import (
    pack_padded_sequence,
    pad_packed_sequence
)


# 1. Encodeur : conserver les représentations de tous les mots
class EncoderLSTM(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_dim,
                 num_layers=1):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size, embedding_dim, padding_idx=PAD_ID
        )

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )

    def forward(self, src, src_lengths):
        embedded = self.embedding(src)

        packed = pack_padded_sequence(
            embedded,
            src_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_outputs, (hidden, cell) = self.lstm(packed)

        # [batch, longueur source, hidden_dim]
        encoder_outputs, _ = pad_packed_sequence(
            packed_outputs,
            batch_first=True,
            total_length=src.size(1)
        )

        return encoder_outputs, hidden, cell


# 2. Attention additive de Bahdanau
class BahdanauAttention(nn.Module):

    def __init__(self, hidden_dim):
        super().__init__()

        self.encoder_projection = nn.Linear(
            hidden_dim, hidden_dim, bias=False
        )
        self.decoder_projection = nn.Linear(
            hidden_dim, hidden_dim, bias=False
        )
        self.score_projection = nn.Linear(
            hidden_dim, 1, bias=False
        )

    def forward(self, query, encoder_outputs, src_mask):
        # query : dernier état caché du décodeur [batch, hidden_dim]
        # src_mask : True pour les positions valides

        energy = torch.tanh(
            self.encoder_projection(encoder_outputs)
            + self.decoder_projection(query).unsqueeze(1)
        )

        # Un score par position française : [batch, longueur source]
        scores = self.score_projection(energy).squeeze(-1)

        # Empêcher l'attention de regarder le padding
        scores = scores.masked_fill(~src_mask, float("-inf"))

        weights = torch.softmax(scores, dim=-1)

        # Somme pondérée des représentations françaises
        context = torch.bmm(
            weights.unsqueeze(1),
            encoder_outputs
        ).squeeze(1)

        return context, weights


# 3. Décodeur : utiliser le contexte pour prédire le prochain token
class DecoderLSTM(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_dim,
                 num_layers=1):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size, embedding_dim, padding_idx=PAD_ID
        )

        self.attention = BahdanauAttention(hidden_dim)

        # Entrée = embedding du token précédent + contexte
        self.lstm = nn.LSTM(
            embedding_dim + hidden_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )

        self.output_layer = nn.Linear(
            hidden_dim * 2,
            vocab_size
        )

    def forward(self, token, hidden, cell,
                encoder_outputs, src_mask):

        embedded = self.embedding(token)

        context, weights = self.attention(
            hidden[-1],
            encoder_outputs,
            src_mask
        )

        lstm_input = torch.cat(
            [embedded, context], dim=-1
        ).unsqueeze(1)

        output, (hidden, cell) = self.lstm(
            lstm_input, (hidden, cell)
        )

        logits = self.output_layer(
            torch.cat([output.squeeze(1), context], dim=-1)
        )

        return logits, hidden, cell, weights


# 4. Modèle complet
class Seq2SeqLSTM(nn.Module):

    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, src_lengths, decoder_input,
                teacher_forcing_ratio=0.5,
                return_attention=False):

        encoder_outputs, hidden, cell = self.encoder(
            src, src_lengths
        )

        src_mask = src.ne(PAD_ID)

        token = decoder_input[:, 0]
        steps = decoder_input.size(1)

        predictions = []
        attention_steps = []

        for t in range(steps):
            logits, hidden, cell, weights = self.decoder(
                token,
                hidden,
                cell,
                encoder_outputs,
                src_mask
            )

            predictions.append(logits)

            if return_attention:
                attention_steps.append(weights)

            if t + 1 < steps:
                if random.random() < teacher_forcing_ratio:
                    token = decoder_input[:, t + 1]
                else:
                    token = logits.argmax(dim=-1)

        predictions = torch.stack(predictions, dim=1)

        if return_attention:
            # [batch, longueur cible, longueur source]
            attention = torch.stack(attention_steps, dim=1)
            return predictions, attention

        return predictions


# 5. Créer le modèle avec les mêmes dimensions que l'expérience 2
random.seed(config["seed"])
torch.manual_seed(config["seed"])

encoder = EncoderLSTM(
    len(vocab_fr),
    config["embedding_dim"],
    config["hidden_dim"],
    config["num_layers"]
)

decoder = DecoderLSTM(
    len(vocab_en),
    config["embedding_dim"],
    config["hidden_dim"],
    config["num_layers"]
)

model = Seq2SeqLSTM(encoder, decoder).to(device)

num_parameters = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("Modèle : LSTM avec attention de Bahdanau")
print("Appareil :", device)
print("Paramètres entraînables :", num_parameters)

Modèle : LSTM avec attention de Bahdanau
Appareil : cuda
Paramètres entraînables : 8874000


# **Verification du modele**

In [6]:
batch = next(iter(train_loader))

# Petit test : 2 phrases et 5 étapes de traduction
src = batch["src"][:2].to(device)
src_lengths = batch["src_lengths"][:2]
decoder_input = batch["decoder_input"][:2, :5].to(device)
target = batch["target"][:2, :5].to(device)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

model.eval()

with torch.no_grad():
    logits, attention = model(
        src,
        src_lengths,
        decoder_input,
        teacher_forcing_ratio=1.0,
        return_attention=True
    )

    loss = criterion(
        logits.reshape(-1, len(vocab_en)),
        target.reshape(-1)
    )

# Vérifier les dimensions et l'absence de valeurs non finies
assert logits.shape == (
    src.size(0), decoder_input.size(1), len(vocab_en)
)
assert attention.shape == (
    src.size(0), decoder_input.size(1), src.size(1)
)
assert torch.isfinite(logits).all().item()
assert torch.isfinite(attention).all().item()
assert torch.isfinite(loss).item()

# Pour chaque token anglais, les poids doivent sommer à 1
assert torch.allclose(
    attention.sum(dim=-1),
    torch.ones_like(attention.sum(dim=-1)),
    atol=1e-5
)

# Les positions PAD doivent recevoir un poids nul
pad_mask = src.eq(PAD_ID).unsqueeze(1).expand_as(attention)
pad_weights = attention.masked_select(pad_mask)

if pad_weights.numel() > 0:
    assert torch.all(pad_weights == 0).item()

print("Dimensions des sorties :", tuple(logits.shape))
print("Dimensions de l'attention :", tuple(attention.shape))
print("Loss initiale :", round(loss.item(), 4))
print("Somme des poids d'attention : OK")
print("Masquage du padding : OK")
print("Modèle avec attention vérifié.")

Dimensions des sorties : (2, 5, 10000)
Dimensions de l'attention : (2, 5, 58)
Loss initiale : 9.2048
Somme des poids d'attention : OK
Masquage du padding : OK
Modèle avec attention vérifié.


# **Préparer l’entraînement**

In [7]:
import time
import json
import random
from pathlib import Path
from tqdm.auto import tqdm

random.seed(config["seed"])
torch.manual_seed(config["seed"])

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config["seed"])

encoder = EncoderLSTM(
    len(vocab_fr),
    config["embedding_dim"],
    config["hidden_dim"],
    config["num_layers"]
)

decoder = DecoderLSTM(
    len(vocab_en),
    config["embedding_dim"],
    config["hidden_dim"],
    config["num_layers"]
)

model = Seq2SeqLSTM(encoder, decoder).to(device)

# Repartir du même ordre de batchs
loaders = make_loaders(prepared, config)
train_loader = loaders["train"]
validation_loader = loaders["validation"]
test_loader = loaders["test"]

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=config["learning_rate"]
)

# Somme des erreurs, sans compter le padding
criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_ID,
    reduction="sum"
)

checkpoint_dir = project_dir / "checkpoints" / "experience_5_bahdanau"
results_dir = project_dir / "results" / "experience_5_bahdanau"

checkpoint_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

print("Appareil :", device)
print("Nombre d'époques :", config["epochs"])
print("Sauvegardes :", checkpoint_dir)

Appareil : cuda
Nombre d'époques : 5
Sauvegardes : /content/drive/MyDrive/Colab Notebooks/nlp_translation/checkpoints/experience_5_bahdanau


# **Définir l’entraînement et la validation**

In [8]:
def train_one_epoch(model, loader):

    model.train()

    total_loss = 0.0
    total_tokens = 0

    for batch in tqdm(loader, desc="Entraînement"):

        src = batch["src"].to(device)
        lengths = batch["src_lengths"]  # Rester sur CPU
        decoder_input = batch["decoder_input"].to(device)
        target = batch["target"].to(device)

        optimizer.zero_grad()

        logits = model(
            src,
            lengths,
            decoder_input,
            teacher_forcing_ratio=config["teacher_forcing_ratio"]
        )

        loss_sum = criterion(
            logits.reshape(-1, len(vocab_en)),
            target.reshape(-1)
        )

        valid_tokens = target.ne(PAD_ID).sum().item()
        loss = loss_sum / valid_tokens

        if not torch.isfinite(loss).item():
            raise RuntimeError("Loss non finie : arrêter et vérifier le modèle.")

        loss.backward()

        # Limiter les gradients trop grands
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        total_loss += loss_sum.detach().item()
        total_tokens += valid_tokens

    return total_loss / total_tokens


@torch.no_grad()
def validate_one_epoch(model, loader):

    model.eval()

    total_loss = 0.0
    total_tokens = 0

    for batch in tqdm(loader, desc="Validation"):

        src = batch["src"].to(device)
        lengths = batch["src_lengths"]
        decoder_input = batch["decoder_input"].to(device)
        target = batch["target"].to(device)

        logits = model(
            src,
            lengths,
            decoder_input,
            teacher_forcing_ratio=1.0
        )

        loss_sum = criterion(
            logits.reshape(-1, len(vocab_en)),
            target.reshape(-1)
        )

        total_loss += loss_sum.item()
        total_tokens += target.ne(PAD_ID).sum().item()

    return total_loss / total_tokens

# **Lancer l’entraînement et sauvegarder**

In [9]:
history = []
best_validation_loss = float("inf")
total_training_seconds = 0.0

num_parameters = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

for epoch in range(1, config["epochs"] + 1):

    print(f"\nÉpoque {epoch}/{config['epochs']}")

    # Synchroniser le GPU pour mesurer le temps
    if device.type == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    train_loss = train_one_epoch(model, train_loader)

    if device.type == "cuda":
        torch.cuda.synchronize()

    training_seconds = time.perf_counter() - start
    total_training_seconds += training_seconds

    validation_loss = validate_one_epoch(
        model,
        validation_loader
    )

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "validation_loss": validation_loss,
        "training_seconds": training_seconds,
        "total_training_seconds": total_training_seconds
    }

    history.append(row)

    print(
        f"Train loss : {train_loss:.4f} | "
        f"Validation loss : {validation_loss:.4f} | "
        f"Temps entraînement : {training_seconds / 60:.2f} min"
    )

    improved = validation_loss < best_validation_loss

    if improved:
        best_validation_loss = validation_loss

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "config": config,
        "vocab_fr": vocab_fr,
        "vocab_en": vocab_en,
        "validation_loss": validation_loss,
        "best_validation_loss": best_validation_loss,
        "num_parameters": num_parameters,
        "history": history,
        "python_rng_state": random.getstate(),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_states": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available() else []
        )
    }

    # Sauvegarde de la dernière époque
    torch.save(
        checkpoint,
        checkpoint_dir / "lstm_last.pt"
    )

    # Sauvegarde du meilleur modèle
    if improved:
        torch.save(
            checkpoint,
            checkpoint_dir / "lstm_best.pt"
        )
        print("Meilleur modèle sauvegardé.")

    # Sauvegarde des résultats après chaque époque
    (results_dir / "lstm_history.json").write_text(
        json.dumps(history, indent=2),
        encoding="utf-8"
    )

print("\nEntraînement terminé.")
print("Meilleure loss de validation :", best_validation_loss)
print("Temps total d'entraînement :", total_training_seconds / 60, "min")


Époque 1/5


Entraînement:   0%|          | 0/3750 [00:00<?, ?it/s]

Validation:   0%|          | 0/250 [00:00<?, ?it/s]

Train loss : 5.6540 | Validation loss : 4.9849 | Temps entraînement : 9.10 min
Meilleur modèle sauvegardé.

Époque 2/5


Entraînement:   0%|          | 0/3750 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/250 [00:00<?, ?it/s]

Train loss : 3.9756 | Validation loss : 4.4434 | Temps entraînement : 9.30 min
Meilleur modèle sauvegardé.

Époque 5/5


Entraînement:   0%|          | 0/3750 [00:00<?, ?it/s]

Validation:   0%|          | 0/250 [00:00<?, ?it/s]

Train loss : 3.6520 | Validation loss : 4.4612 | Temps entraînement : 9.37 min

Entraînement terminé.
Meilleure loss de validation : 4.443373731457546
Temps total d'entraînement : 46.259957197350005 min


# **Charge le meilleur modèle et calculer la loss de test**

In [10]:
checkpoint_dir = (
    project_dir / "checkpoints" / "experience_5_bahdanau"
)
results_dir = (
    project_dir / "results" / "experience_5_bahdanau"
)
results_dir.mkdir(parents=True, exist_ok=True)

checkpoint = torch.load(
    checkpoint_dir / "lstm_best.pt",
    map_location=device,
    weights_only=True
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# La fonction de validation attend une loss avec reduction="sum"
criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_ID,
    reduction="sum"
)

test_loss = validate_one_epoch(model, test_loader)

print("Époque sélectionnée :", checkpoint["epoch"])
print("Loss de test :", round(test_loss, 4))

Validation:   0%|          | 0/250 [00:00<?, ?it/s]

Époque sélectionnée : 4
Loss de test : 4.4277


# **Génére les traductions sans teacher forcing**

In [11]:
SOS_ID = config["special_tokens"]["<SOS>"]
EOS_ID = config["special_tokens"]["<EOS>"]
PAD_ID = config["special_tokens"]["<PAD>"]

id_to_token_en = {
    index: token for token, index in vocab_en.items()
}

# Même limite que dans l'expérience 2
MAX_GENERATION_STEPS = 100


@torch.no_grad()
def translate_batch(src, src_lengths):
    model.eval()

    # L'encodeur retourne maintenant tous les états
    encoder_outputs, hidden, cell = model.encoder(
        src, src_lengths
    )

    src_mask = src.ne(PAD_ID)
    batch_size = src.size(0)

    token = torch.full(
        (batch_size,),
        SOS_ID,
        dtype=torch.long,
        device=src.device
    )

    finished = torch.zeros(
        batch_size,
        dtype=torch.bool,
        device=src.device
    )

    generated_steps = []

    for _ in range(MAX_GENERATION_STEPS):
        # Le décodeur consulte la phrase française via l'attention
        logits, hidden, cell, attention_weights = model.decoder(
            token,
            hidden,
            cell,
            encoder_outputs,
            src_mask
        )

        # Le token prédit devient l'entrée de l'étape suivante
        token = logits.argmax(dim=-1)

        token = torch.where(
            finished,
            torch.full_like(token, EOS_ID),
            token
        )

        generated_steps.append(token.cpu())
        finished = finished | token.eq(EOS_ID)

        if finished.all().item():
            break

    generated_ids = torch.stack(
        generated_steps, dim=1
    ).tolist()

    translations = []

    for sequence in generated_ids:
        words = []

        for index in sequence:
            if index == EOS_ID:
                break

            if index not in {PAD_ID, SOS_ID}:
                words.append(
                    id_to_token_en.get(index, "<UNK>")
                )

        translations.append(" ".join(words))

    return translations, finished.cpu().tolist()

# **Traduire les 2 000 phrases de test**

In [15]:
import pandas as pd

evaluation_rows = []

for batch in tqdm(test_loader, desc="Traduction du test"):

    predictions, ended = translate_batch(
        batch["src"].to(device),
        batch["src_lengths"]
    )

    for source, reference, prediction, length, has_eos in zip(
        batch["source_texts"],
        batch["reference_texts"],
        predictions,
        batch["src_token_counts"].tolist(),
        ended
    ):

        evaluation_rows.append({
            "source": source,
            "reference": reference,
            "prediction": prediction,
            "source_length": length,
            "ended_with_eos": has_eos
        })

translations_df = pd.DataFrame(evaluation_rows)

assert len(translations_df) == 2000

translations_df.to_csv(
    results_dir / "lstm_test_translations.csv",
    index=False,
    encoding="utf-8"
)

display(
    translations_df[
        ["source", "reference", "prediction"]
    ].head(10)
)

print(
    "Phrases ayant atteint la limite sans <EOS> :",
    (~translations_df["ended_with_eos"]).sum()
)

Traduction du test:   0%|          | 0/250 [00:00<?, ?it/s]

,source,reference,prediction
0,- vous étiez en train de vous embrasser à l'ar...,- you were at a bus stop kissing him!,- you were in you to to you to you to you to y...
1,"avec une ironie farceuse, le jeune artiste tch...",with irony and mischief the young czech artist...,"with a <UNK> <UNK> , <UNK> , <UNK> <UNK> <UNK>..."
2,qui va parler au vendeur de voitures ? big fre...,- who's going to talk to the used car salesman?,who will be the <UNK> of the <UNK> ?
3,on vous dit éclairé : votre musique parle d'hu...,"people think you are a great man, that your mu...","you said you , your <UNK> , <UNK> , <UNK> and ..."
4,autres coraux,lace corals,other other
5,"dites, je me demandais...","yeah, there's one man i know who could nail him.","all right , i . . . ."
6,peter et moi étions dans un accident de voitur...,peter and i were in a car crash.,peter and i was in in a car . .
7,ces lois mettent en place une république parle...,these laws establish a parliamentary republic ...,these laws are an important and and and the th...
8,"les activités de suivi, de recherche et d'éval...","monitoring, research and evaluation activities...","activities activities , research and and and a..."
9,lorsque les actes des institutions adoptés ava...,where acts of the institutions prior to 1 may ...,when the member states have been before the 1 ...


Phrases ayant atteint la limite sans <EOS> : 144


# **Calculer le BLEU et sauvegarder les résultats**

In [13]:
%pip install -q sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 13.3 MB/s eta 0:00:00


In [14]:
from sacrebleu.metrics import BLEU

# Comparaison insensible aux majuscules :
# votre préparation utilise des phrases en minuscules.
bleu_metric = BLEU(
    lowercase=True,
    tokenize="13a"
)

bleu_result = bleu_metric.corpus_score(
    translations_df["prediction"].tolist(),
    [translations_df["reference"].tolist()]
)

metrics = {
    "model": "LSTM avec attention de Bahdanau",
    "best_epoch": checkpoint["epoch"],
    "validation_loss": checkpoint["validation_loss"],
    "test_loss": test_loss,
    "bleu": bleu_result.score,
    "bleu_signature": str(bleu_metric.get_signature()),
    "training_minutes": (
    json.loads(
        (results_dir / "lstm_history.json").read_text(
            encoding="utf-8"
        )
    )[-1]["total_training_seconds"] / 60
),
    "num_parameters": checkpoint["num_parameters"],
    "max_generation_steps": MAX_GENERATION_STEPS,
    "test_sentences": len(translations_df),
    "sentences_without_eos": int(
        (~translations_df["ended_with_eos"]).sum()
    )
}

(results_dir / "lstm_metrics.json").write_text(
    json.dumps(metrics, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

display(pd.DataFrame([metrics]))

,model,best_epoch,validation_loss,test_loss,bleu,bleu_signature,training_minutes,num_parameters,max_generation_steps,test_sentences,sentences_without_eos
0,LSTM avec attention de Bahdanau,4,4.443374,4.427734,2.887005,nrefs:1|case:lc|eff:no|tok:13a|smooth:exp|vers...,46.259957,8874000,100,2000,144


In [16]:
import json
import pandas as pd

baseline_path = project_dir / "results" / "lstm_metrics.json"
attention_path = (
    project_dir / "results"
    / "experience_5_bahdanau"
    / "lstm_metrics.json"
)

if not baseline_path.exists():
    print("Résultats de l'expérience 2 introuvables à :")
    print(baseline_path)
    print("Envoie le tableau BLEU de l'expérience 2.")
else:
    baseline = json.loads(
        baseline_path.read_text(encoding="utf-8")
    )
    attention = json.loads(
        attention_path.read_text(encoding="utf-8")
    )

    columns = [
        "model",
        "best_epoch",
        "test_loss",
        "bleu",
        "num_parameters",
        "sentences_without_eos"
    ]

    comparison = pd.DataFrame([baseline, attention])
    display(comparison[columns])

    # Vérifier que les paramètres de mesure sont identiques
    same_protocol = all(
        baseline.get(key) == attention.get(key)
        for key in [
            "bleu_signature",
            "max_generation_steps",
            "test_sentences"
        ]
    )

    if same_protocol:
        gain = attention["bleu"] - baseline["bleu"]
        print(f"Écart de BLEU : {gain:+.3f} points")
        print("Comparaison valable si les phrases de test sont identiques.")
    else:
        print(
            "Les paramètres d'évaluation diffèrent : "
            "vérifier le protocole avant d'interpréter l'écart."
        )

,model,best_epoch,test_loss,bleu,num_parameters,sentences_without_eos
0,LSTM sans attention,5,4.792028,1.240395,5920528,22
1,LSTM avec attention de Bahdanau,4,4.427734,2.887005,8874000,144


Écart de BLEU : +1.647 points
Comparaison valable si les phrases de test sont identiques.


In [17]:
# Examiner les données d'entraînement réellement reçues par le modèle
batch_check = next(iter(train_loader))

id_to_fr = {i: token for token, i in vocab_fr.items()}
id_to_en = {i: token for token, i in vocab_en.items()}

pad_id = config["special_tokens"]["<PAD>"]
sos_id = config["special_tokens"]["<SOS>"]
eos_id = config["special_tokens"]["<EOS>"]
unk_id = config["special_tokens"]["<UNK>"]


def decode_ids(ids, vocabulary):
    return " ".join(
        vocabulary.get(int(i), "<?>")
        for i in ids
        if int(i) != pad_id
    )


for i in range(min(3, batch_check["src"].size(0))):
    print(f"\n===== PAIRE {i + 1} =====")

    print("FR original :", batch_check["source_texts"][i])
    print("EN référence :", batch_check["reference_texts"][i])

    print(
        "FR encodé :",
        decode_ids(batch_check["src"][i], id_to_fr)
    )
    print(
        "Entrée décodeur :",
        decode_ids(batch_check["decoder_input"][i], id_to_en)
    )
    print(
        "Cible attendue :",
        decode_ids(batch_check["target"][i], id_to_en)
    )

    decoder_ids = batch_check["decoder_input"][i]
    target_ids = batch_check["target"][i]

    decoder_ids = decoder_ids[decoder_ids != pad_id].tolist()
    target_ids = target_ids[target_ids != pad_id].tolist()

    shift_ok = (
        len(decoder_ids) > 0
        and len(target_ids) > 0
        and decoder_ids[0] == sos_id
        and target_ids[-1] == eos_id
        and decoder_ids[1:] == target_ids[:-1]
    )

    print("Décalage SOS / cible / EOS correct :", shift_ok)


# Mesurer les mots inconnus dans l'ensemble d'entraînement
counts = {
    "fr_tokens": 0, "fr_unk": 0,
    "en_tokens": 0, "en_unk": 0
}

for batch_check in train_loader:
    for key, prefix in [("src", "fr"), ("target", "en")]:
        ids = batch_check[key]

        valid = (
            ids.ne(pad_id)
            & ids.ne(sos_id)
            & ids.ne(eos_id)
        )

        counts[f"{prefix}_tokens"] += valid.sum().item()
        counts[f"{prefix}_unk"] += (
            ids.eq(unk_id) & valid
        ).sum().item()

for prefix, language in [("fr", "Français"), ("en", "Anglais")]:
    rate = (
        100 * counts[f"{prefix}_unk"]
        / max(counts[f"{prefix}_tokens"], 1)
    )
    print(f"\nTokens <UNK> — {language} : {rate:.2f} %")

print("\nTeacher forcing :", config["teacher_forcing_ratio"])
print("Learning rate :", config["learning_rate"])


===== PAIRE 1 =====
FR original : c) les autorités en matière de concurrence de la partie s'engagent, dans la mise en oeuvre de leurs propres mesures d'application:
EN référence : (c) the competition authorities of the requested party agree that in conducting their own enforcement activities, they will:
FR encodé : c ) les autorités en matière de concurrence de la partie s'engagent , dans la mise en oeuvre de leurs propres mesures d'application : <EOS>
Entrée décodeur : <SOS> ( c ) the competition authorities of the requested party agree that in conducting their own enforcement activities , they will :
Cible attendue : ( c ) the competition authorities of the requested party agree that in conducting their own enforcement activities , they will : <EOS>
Décalage SOS / cible / EOS correct : True

===== PAIRE 2 =====
FR original : sean ramsey n'est pas celui que vous croyez.
EN référence : sean ramsey is not who you think he is.
FR encodé : <UNK> <UNK> n'est pas celui que vous croyez . <E

In [18]:
import copy

# Copie indépendante : le modèle principal reste intact
diagnostic_model = copy.deepcopy(model).to(device)

diagnostic_optimizer = torch.optim.Adam(
    diagnostic_model.parameters(),
    lr=0.001
)

diagnostic_criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_ID
)

# Toujours les mêmes 8 phrases pour ce test
diagnostic_batch = next(iter(train_loader))

src_diag = diagnostic_batch["src"].to(device)
lengths_diag = diagnostic_batch["src_lengths"]
inputs_diag = diagnostic_batch["decoder_input"].to(device)
targets_diag = diagnostic_batch["target"].to(device)

diagnostic_model.train()

for step in range(201):
    diagnostic_optimizer.zero_grad()

    logits_diag = diagnostic_model(
        src_diag,
        lengths_diag,
        inputs_diag,
        teacher_forcing_ratio=1.0
    )

    loss_diag = diagnostic_criterion(
        logits_diag.reshape(-1, len(vocab_en)),
        targets_diag.reshape(-1)
    )

    if step % 50 == 0:
        print(f"Étape {step:3d} | Loss : {loss_diag.item():.4f}")

    if step == 200:
        break

    loss_diag.backward()

    nn.utils.clip_grad_norm_(
        diagnostic_model.parameters(),
        max_norm=1.0
    )

    diagnostic_optimizer.step()

Étape   0 | Loss : 3.9067
Étape  50 | Loss : 0.0457
Étape 100 | Loss : 0.0072
Étape 150 | Loss : 0.0038
Étape 200 | Loss : 0.0024


In [19]:
@torch.no_grad()
def translate_diagnostic():
    diagnostic_model.eval()

    enc_outputs, hidden, cell = diagnostic_model.encoder(
        src_diag, lengths_diag
    )

    mask = src_diag.ne(PAD_ID)

    token = torch.full(
        (src_diag.size(0),),
        SOS_ID,
        dtype=torch.long,
        device=device
    )

    finished = torch.zeros(
        src_diag.size(0),
        dtype=torch.bool,
        device=device
    )

    generated = []

    for _ in range(MAX_GENERATION_STEPS):
        logits, hidden, cell, _ = diagnostic_model.decoder(
            token, hidden, cell, enc_outputs, mask
        )

        token = logits.argmax(dim=-1)
        token = torch.where(
            finished,
            torch.full_like(token, EOS_ID),
            token
        )

        generated.append(token.cpu())
        finished |= token.eq(EOS_ID)

        if finished.all().item():
            break

    sequences = torch.stack(generated, dim=1).tolist()
    texts = []

    for sequence in sequences:
        words = []

        for index in sequence:
            if index == EOS_ID:
                break
            if index not in {PAD_ID, SOS_ID}:
                words.append(id_to_en.get(index, "<UNK>"))

        texts.append(" ".join(words))

    return texts


diagnostic_predictions = translate_diagnostic()

for i in range(min(3, len(diagnostic_predictions))):
    print(f"\n===== PHRASE {i + 1} =====")
    print("FR :", diagnostic_batch["source_texts"][i])
    print("Référence :", diagnostic_batch["reference_texts"][i])
    print("Prédiction :", diagnostic_predictions[i])


===== PHRASE 1 =====
FR : je suis occupé à nourrir ima.
Référence : i've been busy feeding ima.
Prédiction : i've been busy feeding <UNK> .

===== PHRASE 2 =====
FR : ce contrôle a lieu en permanence.
Référence : this monitoring is going on constantly.
Prédiction : this monitoring is going on constantly .

===== PHRASE 3 =====
FR : après l'élargissement historique à dix pays d'europe centrale et orientale et du bassin méditerranéen en 2004, la roumanie et la bulgarie sont maintenant entrées dans l'ue en 2007.
Référence : after the historic enlargement by 10 countries of central and eastern europe and the mediterranean in 2004, romania and bulgaria have now joined the eu at the beginning of 2007.
Prédiction : after the historic enlargement by 10 countries of central and eastern europe and the mediterranean in 2004 , romania and bulgaria have now joined the eu at the beginning of 2007 .
